In [ ]:
# Fabric Notebook: NB_60_POI_DATA_QUALITY
# Default Lakehouse: lh_vv_silver (MANDATORY)
# Gold targets use four-part workspace.lakehouse.schema.table names.
# READ : Silver + Gold
# WRITE: Gold active status + Gold outbound sync staging tables
# NOTE : DQ is evaluated with TEMP VIEWs only; poi_quality_current / poi_quality_issue are not written.

run_id = globals().get("run_id", "DEV_RUN")
STAGE = "_stage_poi_language"

spark.sql(f"CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS SELECT DISTINCT poi_id,poi_group_id,lang,source_name,source_id FROM {STAGE} WHERE run_id='{run_id}'")

# -----------------------------------------------------------------------------
# Impacted POIs for this run.
# DQ is intentionally ephemeral: no writes to poi_quality_issue/current.
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_impacted_poi_ids AS
SELECT DISTINCT poi_id
FROM v2_impacted_poi
WHERE poi_id IS NOT NULL
""")

# -----------------------------------------------------------------------------
# DQ rule sources.
# Materialize rule sets first instead of using correlated subqueries inside one
# large UNION query. This is easier to debug and more stable in Fabric Spark.
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_current_entity AS
SELECT DISTINCT poi_id
FROM poi_entity
WHERE _is_current = true
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_duplicate_current_entity AS
SELECT poi_id
FROM poi_entity
WHERE _is_current = true
GROUP BY poi_id
HAVING COUNT(*) > 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_current_enrichment AS
SELECT DISTINCT poi_id
FROM poi_enrichment
WHERE is_current = true
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_duplicate_current_enrichment AS
SELECT poi_id
FROM poi_enrichment
WHERE is_current = true
GROUP BY poi_id
HAVING COUNT(*) > 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_source_mapped AS
SELECT DISTINCT poi_id
FROM poi_source_map
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_active_photo AS
SELECT DISTINCT poi_id
FROM poi_media
WHERE is_active = true
  AND media_type = 'photo'
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_amenity_present AS
SELECT DISTINCT poi_id
FROM poi_amenity_current
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_amenity_meaningful AS
SELECT DISTINCT poi_id
FROM poi_amenity_current
WHERE NULLIF(TRIM(resolved_amenities_json), '') IS NOT NULL
   OR NULLIF(TRIM(ext_attributes_json), '') IS NOT NULL
   OR NULLIF(TRIM(place_types_json), '') IS NOT NULL
   OR NULLIF(TRIM(primary_type), '') IS NOT NULL
""")

# Critical / Error / Warning rules
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_quality_detected AS

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ001' AS rule_id,
    'CRITICAL' AS severity,
    'missing_current_entity' AS reason
FROM v2_impacted_poi i
LEFT ANTI JOIN v2_current_entity e
    ON e.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ002' AS rule_id,
    'CRITICAL' AS severity,
    'duplicate_current_entity' AS reason
FROM v2_impacted_poi i
INNER JOIN v2_duplicate_current_entity x
    ON x.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ003' AS rule_id,
    'CRITICAL' AS severity,
    'duplicate_current_enrichment' AS reason
FROM v2_impacted_poi i
INNER JOIN v2_duplicate_current_enrichment x
    ON x.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ004' AS rule_id,
    'ERROR' AS severity,
    'missing_source_map' AS reason
FROM v2_impacted_poi i
LEFT ANTI JOIN v2_source_mapped s
    ON s.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ101' AS rule_id,
    'WARNING' AS severity,
    'missing_enrichment' AS reason
FROM v2_impacted_poi i
LEFT ANTI JOIN v2_current_enrichment e
    ON e.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ102' AS rule_id,
    'WARNING' AS severity,
    'missing_active_media' AS reason
FROM v2_impacted_poi i
LEFT ANTI JOIN v2_active_photo m
    ON m.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ103' AS rule_id,
    'WARNING' AS severity,
    'missing_amenity_current' AS reason
FROM v2_impacted_poi i
LEFT ANTI JOIN v2_amenity_present a
    ON a.poi_id = i.poi_id

UNION ALL

SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    'DQ104' AS rule_id,
    'WARNING' AS severity,
    'empty_amenity_payload' AS reason
FROM v2_impacted_poi i
INNER JOIN v2_amenity_present a
    ON a.poi_id = i.poi_id
LEFT ANTI JOIN v2_amenity_meaningful m
    ON m.poi_id = i.poi_id
""")

# -----------------------------------------------------------------------------
# Ephemeral DQ result for this run.
# No persistent poi_quality_issue / poi_quality_current tables are created,
# updated, or read by NB60.
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_quality_current AS
SELECT
    i.poi_id,
    i.poi_group_id,
    i.lang,
    CASE
        WHEN SUM(CASE WHEN q.severity='CRITICAL' THEN 1 ELSE 0 END) > 0 THEN 'INVALID'
        WHEN SUM(CASE WHEN q.severity='ERROR' THEN 1 ELSE 0 END) > 0 THEN 'INCOMPLETE'
        WHEN SUM(CASE WHEN q.severity='WARNING' THEN 1 ELSE 0 END) > 0 THEN 'INCOMPLETE'
        ELSE 'VALID'
    END AS quality_status,
    SUM(CASE WHEN q.severity='CRITICAL' THEN 1 ELSE 0 END) AS critical_count,
    SUM(CASE WHEN q.severity='ERROR' THEN 1 ELSE 0 END) AS error_count,
    SUM(CASE WHEN q.severity='WARNING' THEN 1 ELSE 0 END) AS warning_count
FROM v2_impacted_poi i
LEFT JOIN v2_quality_detected q
    ON q.poi_id = i.poi_id
GROUP BY i.poi_id,i.poi_group_id,i.lang
""")

# Deactivate ONLY critical-invalid POIs. Warnings/errors do not automatically deactivate.
spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
USING (
    SELECT q.poi_id
    FROM v2_quality_current q
    INNER JOIN v2_impacted_poi_ids i
        ON i.poi_id = q.poi_id
    WHERE q.quality_status='INVALID'
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.is_active=false,tgt.updated_at=current_timestamp()
""")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi tgt
USING (
    SELECT q.poi_id
    FROM v2_quality_current q
    INNER JOIN v2_impacted_poi_ids i
        ON i.poi_id = q.poi_id
    WHERE q.quality_status='INVALID'
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.is_active=false,tgt.updated_at=current_timestamp()
""")

# Blacklist/freeze V2 behavior:
# Apply policy by source identity / poi_group_id so all languages of the same source POI are affected.
# For this implementation we keep existing CSV policy input, but avoid driver collect.
# Materialize the exact blacklist-propagated POI set so outbound staging can
# include every Gold row changed by this policy, not only the original run POIs.
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_blacklist_affected_poi_ids AS
SELECT CAST(NULL AS STRING) AS poi_id
WHERE 1 = 0
""")

try:
    blacklist = (
        spark.read
        .option("header","false")
        .csv("Files/pipe_poi_google/blacklist.csv")
        .selectExpr("trim(_c0) as policy_value")
        .filter("policy_value is not null and policy_value<>''")
    )
    blacklist.createOrReplaceTempView("v2_blacklist")

    # Supports existing poi_id list; if the file is migrated to source_id /
    # poi_group_id, extend this join.
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW v2_blacklist_affected_poi_ids AS
    SELECT DISTINCT sm2.poi_id
    FROM poi_source_map sm1
    JOIN v2_blacklist b
      ON b.policy_value = sm1.poi_id
    JOIN poi_source_map sm2
      ON COALESCE(sm2.poi_group_id,'') = COALESCE(sm1.poi_group_id,'')
     AND sm2.source_name = sm1.source_name
     AND sm2.source_id = sm1.source_id
    WHERE sm2.poi_id IS NOT NULL
    """)

    spark.sql("""
    MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
    USING v2_blacklist_affected_poi_ids src
      ON tgt.poi_id = src.poi_id
    WHEN MATCHED THEN UPDATE SET
      tgt.is_active = false,
      tgt.updated_at = current_timestamp()
    """)
except Exception as e:
    print(f"Blacklist skipped: {e}")

# Hard invariants: evaluate directly from this run's TEMP DQ result.
critical_count = spark.sql("""
SELECT COUNT(*) AS c
FROM v2_quality_detected
WHERE severity='CRITICAL'
""").first()["c"]
if critical_count > 0:
    raise Exception(
        f"DATA_QUALITY_CRITICAL: {critical_count} critical issues. Watermark must not advance."
    )


# =============================================================================
# OUTBOUND SYNC STAGING - RESTORED IN NB60
# =============================================================================
# Only executed after all NB60 hard DQ invariants pass.
#
# Shared STG is a latest-state buffer:
#   poi_master              -> unique poi_id
#   poi_detail              -> unique poi_id
#   poi_enrichment_display  -> unique poi_id
#   poi_review              -> unique review_id
#
# NB00-NB60 never writes COPY STG. PostgreSQL sync PREPARE owns COPY STG.
#
# IMPORTANT:
# - Full CURRENT Gold state is staged, not a partial patch.
# - MERGE keeps at most one row per sync key.
# - Every producer write generates a fresh change_token.
# - Blacklist-propagated POIs are included in addition to run-impacted POIs.
# =============================================================================

from pyspark.sql import functions as F

GOLD_DB = "sgr_visitvn_dev.lh_vv_gold.dbo"
PRODUCER_FLOW = "POI_MAIN_PIPELINE"

SYNC_ENTITY_CONFIG = {
    "poi_master": {
        "gold": f"{GOLD_DB}.poi_master",
        "shared": f"{GOLD_DB}.stg_poi_master_sync",
        "sync_key": "poi_id",
        "affected_key": "poi_id",
        "affected_view": "v2_sync_impacted_poi_ids",
    },
    "poi_detail": {
        "gold": f"{GOLD_DB}.poi_detail",
        "shared": f"{GOLD_DB}.stg_poi_detail_sync",
        "sync_key": "poi_id",
        "affected_key": "poi_id",
        "affected_view": "v2_sync_impacted_poi_ids",
    },
    "poi_enrichment_display": {
        "gold": f"{GOLD_DB}.poi_enrichment_display",
        "shared": f"{GOLD_DB}.stg_poi_enrichment_display_sync",
        "sync_key": "poi_id",
        "affected_key": "poi_id",
        "affected_view": "v2_sync_impacted_poi_ids",
    },
    "poi_review": {
        "gold": f"{GOLD_DB}.poi_review",
        "shared": f"{GOLD_DB}.stg_poi_review_sync",
        "sync_key": "review_id",
        # Review keys are derived set-wise from reviews belonging to affected POIs.
        "affected_key": "poi_id",
        "affected_view": "v2_sync_impacted_poi_ids",
    },
}

SHARED_META = {
    "change_token",
    "producer_flow",
    "producer_run_id",
    "source_event_id",
    "source_version",
    "staged_at",
}


def _qcol(name):
    return f"`{name.replace('`', '``')}`"


def _table_exists(name):
    try:
        _ = spark.table(name).schema
        return True
    except Exception:
        return False


def _duplicate_key_groups(table_name, key):
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM (
                SELECT {_qcol(key)}
                FROM {table_name}
                WHERE {_qcol(key)} IS NOT NULL
                GROUP BY {_qcol(key)}
                HAVING COUNT(*) > 1
            ) d
        """).first()["c"]
    )


# Final affected POIs = original run POIs + every POI changed by blacklist
# propagation. DQ deactivation itself is already a subset of original impacted
# POIs, so it is covered by v2_impacted_poi_ids.
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_sync_impacted_poi_ids AS
SELECT DISTINCT poi_id
FROM (
    SELECT poi_id
    FROM v2_impacted_poi_ids

    UNION

    SELECT poi_id
    FROM v2_blacklist_affected_poi_ids
) x
WHERE poi_id IS NOT NULL
""")

sync_impacted_poi_count = spark.sql("""
SELECT COUNT(*) AS c
FROM v2_sync_impacted_poi_ids
""").first()["c"]

print(
    f"Outbound sync affected POIs: {sync_impacted_poi_count}"
)

sync_stage_result = {}

for entity, cfg in SYNC_ENTITY_CONFIG.items():
    gold = cfg["gold"]
    shared = cfg["shared"]
    sync_key = cfg["sync_key"]
    affected_key = cfg["affected_key"]
    affected_view = cfg["affected_view"]

    if not _table_exists(gold):
        raise Exception(
            f"SYNC_GOLD_TABLE_NOT_FOUND: entity={entity}, table={gold}"
        )

    if not _table_exists(shared):
        raise Exception(
            f"SYNC_SHARED_STAGING_NOT_FOUND: entity={entity}, table={shared}. "
            "Run NB_POI_CHANGE_STAGING_SETUP_V4 first."
        )

    gold_cols = spark.table(gold).columns
    shared_cols = set(spark.table(shared).columns)

    if sync_key not in gold_cols:
        raise Exception(
            f"SYNC_KEY_MISSING_IN_GOLD: entity={entity}, key={sync_key}"
        )

    if affected_key not in gold_cols:
        raise Exception(
            f"AFFECTED_KEY_MISSING_IN_GOLD: "
            f"entity={entity}, key={affected_key}"
        )

    required_shared = set(gold_cols) | SHARED_META
    missing_shared = sorted(required_shared - shared_cols)

    if missing_shared:
        raise Exception(
            f"SYNC_SHARED_SCHEMA_MISMATCH: entity={entity}, "
            f"table={shared}, missing={missing_shared}"
        )

    # Current full Gold snapshot for this run's affected POIs.
    #
    # For poi_review this intentionally returns ALL current reviews belonging
    # to affected POIs, then MERGEs each review by review_id.
    current_snapshot = (
        spark.table(gold).alias("g")
        .join(
            spark.table(affected_view)
            .select(F.col("poi_id").alias("_affected_poi_id"))
            .distinct()
            .alias("a"),
            F.col(f"g.{affected_key}") == F.col("a._affected_poi_id"),
            "inner",
        )
        .select(*[F.col(f"g.{c}").alias(c) for c in gold_cols])
        .withColumn("change_token", F.expr("uuid()"))
        .withColumn("producer_flow", F.lit(PRODUCER_FLOW))
        .withColumn("producer_run_id", F.lit(str(run_id)))
        .withColumn("source_event_id", F.lit(None).cast("string"))
        .withColumn("source_version", F.lit(None).cast("long"))
        .withColumn("staged_at", F.current_timestamp())
    )

    # Defensive source uniqueness. Shared STG is required to contain at most
    # one row per sync key.
    duplicate_source_groups = (
        current_snapshot
        .groupBy(sync_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_source_keys = (
        current_snapshot
        .filter(F.col(sync_key).isNull())
        .count()
    )

    if null_source_keys > 0:
        raise Exception(
            f"SYNC_SOURCE_NULL_KEY: entity={entity}, "
            f"sync_key={sync_key}, count={null_source_keys}"
        )

    if duplicate_source_groups > 0:
        raise Exception(
            f"SYNC_SOURCE_DUPLICATE_KEY: entity={entity}, "
            f"sync_key={sync_key}, groups={duplicate_source_groups}"
        )

    source_count = current_snapshot.count()

    if source_count == 0:
        sync_stage_result[entity] = {
            "sync_key": sync_key,
            "source_rows": 0,
            "status": "NO_DATA",
        }
        print(
            f"Shared STG skip: entity={entity}, sync_key={sync_key}, rows=0"
        )
        continue

    stage_view = f"v2_sync_stage_{entity}"
    current_snapshot.createOrReplaceTempView(stage_view)

    write_cols = gold_cols + [
        "change_token",
        "producer_flow",
        "producer_run_id",
        "source_event_id",
        "source_version",
        "staged_at",
    ]

    update_cols = [
        c for c in write_cols
        if c != sync_key
    ]

    spark.sql(f"""
    MERGE INTO {shared} tgt
    USING {stage_view} src
      ON tgt.{_qcol(sync_key)} = src.{_qcol(sync_key)}

    WHEN MATCHED THEN UPDATE SET
      {", ".join(
          f"tgt.{_qcol(c)} = src.{_qcol(c)}"
          for c in update_cols
      )}

    WHEN NOT MATCHED THEN INSERT (
      {", ".join(_qcol(c) for c in write_cols)}
    )
    VALUES (
      {", ".join(f"src.{_qcol(c)}" for c in write_cols)}
    )
    """)

    # Post-MERGE hard invariant.
    duplicate_shared_groups = _duplicate_key_groups(
        shared,
        sync_key,
    )

    if duplicate_shared_groups > 0:
        raise Exception(
            f"SYNC_SHARED_DUPLICATE_KEY_AFTER_MERGE: entity={entity}, "
            f"sync_key={sync_key}, groups={duplicate_shared_groups}"
        )

    sync_stage_result[entity] = {
        "sync_key": sync_key,
        "source_rows": source_count,
        "status": "STAGED",
    }

    print(
        f"Shared STG ready: entity={entity}, "
        f"sync_key={sync_key}, rows={source_count}"
    )


print(
    "NB60 outbound staging result: "
    + str(sync_stage_result)
)

notebookutils.notebook.exit("SUCCESS")


In [ ]:
# # Fabric Notebook: NB_60_POI_DATA_QUALITY
# # Default Lakehouse: lh_vv_silver (MANDATORY)
# # Gold targets use four-part workspace.lakehouse.schema.table names.
# # READ : Silver + Gold
# # WRITE: Silver quality tables + Gold active status

# run_id = globals().get("run_id", "DEV_RUN")
# STAGE = "_stage_poi_language"

# spark.sql("""
# CREATE TABLE IF NOT EXISTS poi_quality_issue (
#  issue_id STRING,run_id STRING,poi_id STRING,poi_group_id STRING,lang STRING,
#  rule_id STRING,severity STRING,status STRING,reason STRING,
#  detected_at TIMESTAMP,resolved_at TIMESTAMP
# ) USING DELTA
# """)

# spark.sql("""
# CREATE TABLE IF NOT EXISTS poi_quality_current (
#  poi_id STRING,poi_group_id STRING,lang STRING,quality_status STRING,
#  critical_count INT,error_count INT,warning_count INT,last_checked_at TIMESTAMP,run_id STRING
# ) USING DELTA
# """)

# spark.sql(f"CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS SELECT DISTINCT poi_id,poi_group_id,lang,source_name,source_id FROM {STAGE} WHERE run_id='{run_id}'")

# # -----------------------------------------------------------------------------
# # Resolve previous OPEN issues for impacted POIs.
# #
# # Fabric Delta does not support subqueries in UPDATE conditions:
# #   UPDATE ... WHERE poi_id IN (SELECT ...)
# #
# # Use MERGE instead.
# # -----------------------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_impacted_poi_ids AS
# SELECT DISTINCT poi_id
# FROM v2_impacted_poi
# WHERE poi_id IS NOT NULL
# """)

# spark.sql("""
# MERGE INTO poi_quality_issue tgt
# USING v2_impacted_poi_ids src
# ON tgt.poi_id = src.poi_id
# AND tgt.status = 'OPEN'
# WHEN MATCHED THEN UPDATE SET
#     tgt.status = 'RESOLVED',
#     tgt.resolved_at = current_timestamp()
# """)

# # -----------------------------------------------------------------------------
# # DQ rule sources.
# # Materialize rule sets first instead of using correlated subqueries inside one
# # large UNION query. This is easier to debug and more stable in Fabric Spark.
# # -----------------------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_current_entity AS
# SELECT DISTINCT poi_id
# FROM poi_entity
# WHERE _is_current = true
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_duplicate_current_entity AS
# SELECT poi_id
# FROM poi_entity
# WHERE _is_current = true
# GROUP BY poi_id
# HAVING COUNT(*) > 1
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_current_enrichment AS
# SELECT DISTINCT poi_id
# FROM poi_enrichment
# WHERE is_current = true
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_duplicate_current_enrichment AS
# SELECT poi_id
# FROM poi_enrichment
# WHERE is_current = true
# GROUP BY poi_id
# HAVING COUNT(*) > 1
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_source_mapped AS
# SELECT DISTINCT poi_id
# FROM poi_source_map
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_active_photo AS
# SELECT DISTINCT poi_id
# FROM poi_media
# WHERE is_active = true
#   AND media_type = 'photo'
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_amenity_present AS
# SELECT DISTINCT poi_id
# FROM poi_amenity_current
# """)

# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_amenity_meaningful AS
# SELECT DISTINCT poi_id
# FROM poi_amenity_current
# WHERE NULLIF(TRIM(resolved_amenities_json), '') IS NOT NULL
#    OR NULLIF(TRIM(ext_attributes_json), '') IS NOT NULL
#    OR NULLIF(TRIM(place_types_json), '') IS NOT NULL
#    OR NULLIF(TRIM(primary_type), '') IS NOT NULL
# """)

# # Critical / Error / Warning rules
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_quality_detected AS

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ001' AS rule_id,
#     'CRITICAL' AS severity,
#     'missing_current_entity' AS reason
# FROM v2_impacted_poi i
# LEFT ANTI JOIN v2_current_entity e
#     ON e.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ002' AS rule_id,
#     'CRITICAL' AS severity,
#     'duplicate_current_entity' AS reason
# FROM v2_impacted_poi i
# INNER JOIN v2_duplicate_current_entity x
#     ON x.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ003' AS rule_id,
#     'CRITICAL' AS severity,
#     'duplicate_current_enrichment' AS reason
# FROM v2_impacted_poi i
# INNER JOIN v2_duplicate_current_enrichment x
#     ON x.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ004' AS rule_id,
#     'ERROR' AS severity,
#     'missing_source_map' AS reason
# FROM v2_impacted_poi i
# LEFT ANTI JOIN v2_source_mapped s
#     ON s.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ101' AS rule_id,
#     'WARNING' AS severity,
#     'missing_enrichment' AS reason
# FROM v2_impacted_poi i
# LEFT ANTI JOIN v2_current_enrichment e
#     ON e.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ102' AS rule_id,
#     'WARNING' AS severity,
#     'missing_active_media' AS reason
# FROM v2_impacted_poi i
# LEFT ANTI JOIN v2_active_photo m
#     ON m.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ103' AS rule_id,
#     'WARNING' AS severity,
#     'missing_amenity_current' AS reason
# FROM v2_impacted_poi i
# LEFT ANTI JOIN v2_amenity_present a
#     ON a.poi_id = i.poi_id

# UNION ALL

# SELECT
#     i.poi_id,
#     i.poi_group_id,
#     i.lang,
#     'DQ104' AS rule_id,
#     'WARNING' AS severity,
#     'empty_amenity_payload' AS reason
# FROM v2_impacted_poi i
# INNER JOIN v2_amenity_present a
#     ON a.poi_id = i.poi_id
# LEFT ANTI JOIN v2_amenity_meaningful m
#     ON m.poi_id = i.poi_id
# """)

# # -----------------------------------------------------------------------------
# # Insert current-run issues idempotently.
# # issue_id is deterministic by run_id + poi_id + rule_id.
# # -----------------------------------------------------------------------------
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_quality_issue_src AS
# SELECT
#     sha2(concat('{run_id}','|',poi_id,'|',rule_id),256) AS issue_id,
#     '{run_id}' AS run_id,
#     poi_id,
#     poi_group_id,
#     lang,
#     rule_id,
#     severity,
#     'OPEN' AS status,
#     reason,
#     current_timestamp() AS detected_at,
#     CAST(NULL AS TIMESTAMP) AS resolved_at
# FROM v2_quality_detected
# """)

# spark.sql("""
# MERGE INTO poi_quality_issue tgt
# USING v2_quality_issue_src src
# ON tgt.issue_id = src.issue_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.run_id = src.run_id,
#     tgt.poi_id = src.poi_id,
#     tgt.poi_group_id = src.poi_group_id,
#     tgt.lang = src.lang,
#     tgt.rule_id = src.rule_id,
#     tgt.severity = src.severity,
#     tgt.status = 'OPEN',
#     tgt.reason = src.reason,
#     tgt.detected_at = src.detected_at,
#     tgt.resolved_at = NULL

# WHEN NOT MATCHED THEN INSERT (
#     issue_id,
#     run_id,
#     poi_id,
#     poi_group_id,
#     lang,
#     rule_id,
#     severity,
#     status,
#     reason,
#     detected_at,
#     resolved_at
# )
# VALUES (
#     src.issue_id,
#     src.run_id,
#     src.poi_id,
#     src.poi_group_id,
#     src.lang,
#     src.rule_id,
#     src.severity,
#     src.status,
#     src.reason,
#     src.detected_at,
#     src.resolved_at
# )
# """)

# spark.sql(f"""
# MERGE INTO poi_quality_current tgt
# USING (
#  SELECT i.poi_id,i.poi_group_id,i.lang,
#    CASE WHEN SUM(CASE WHEN q.severity='CRITICAL' THEN 1 ELSE 0 END)>0 THEN 'INVALID'
#         WHEN SUM(CASE WHEN q.severity='ERROR' THEN 1 ELSE 0 END)>0 THEN 'INCOMPLETE'
#         WHEN SUM(CASE WHEN q.severity='WARNING' THEN 1 ELSE 0 END)>0 THEN 'INCOMPLETE'
#         ELSE 'VALID' END quality_status,
#    SUM(CASE WHEN q.severity='CRITICAL' THEN 1 ELSE 0 END) critical_count,
#    SUM(CASE WHEN q.severity='ERROR' THEN 1 ELSE 0 END) error_count,
#    SUM(CASE WHEN q.severity='WARNING' THEN 1 ELSE 0 END) warning_count,
#    current_timestamp() last_checked_at,'{run_id}' run_id
#  FROM v2_impacted_poi i LEFT JOIN v2_quality_detected q ON q.poi_id=i.poi_id
#  GROUP BY i.poi_id,i.poi_group_id,i.lang
# ) src ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET tgt.poi_group_id=src.poi_group_id,tgt.lang=src.lang,tgt.quality_status=src.quality_status,tgt.critical_count=src.critical_count,tgt.error_count=src.error_count,tgt.warning_count=src.warning_count,tgt.last_checked_at=src.last_checked_at,tgt.run_id=src.run_id
# WHEN NOT MATCHED THEN INSERT (
#     poi_id,
#     poi_group_id,
#     lang,
#     quality_status,
#     critical_count,
#     error_count,
#     warning_count,
#     last_checked_at,
#     run_id
# )
# VALUES (
#     src.poi_id,
#     src.poi_group_id,
#     src.lang,
#     src.quality_status,
#     src.critical_count,
#     src.error_count,
#     src.warning_count,
#     src.last_checked_at,
#     src.run_id
# )
# """)

# # Deactivate ONLY critical-invalid POIs. Warnings/errors do not automatically deactivate.
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
# USING (
#     SELECT q.poi_id
#     FROM poi_quality_current q
#     INNER JOIN v2_impacted_poi_ids i
#         ON i.poi_id = q.poi_id
#     WHERE q.quality_status='INVALID'
# ) src
# ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET tgt.is_active=false,tgt.updated_at=current_timestamp()
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi tgt
# USING (
#     SELECT q.poi_id
#     FROM poi_quality_current q
#     INNER JOIN v2_impacted_poi_ids i
#         ON i.poi_id = q.poi_id
#     WHERE q.quality_status='INVALID'
# ) src
# ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET tgt.is_active=false,tgt.updated_at=current_timestamp()
# """)

# # Blacklist/freeze V2 behavior:
# # Apply policy by source identity / poi_group_id so all languages of the same source POI are affected.
# # For this implementation we keep existing CSV policy input, but avoid driver collect.
# # Materialize the exact blacklist-propagated POI set so outbound staging can
# # include every Gold row changed by this policy, not only the original run POIs.
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_blacklist_affected_poi_ids AS
# SELECT CAST(NULL AS STRING) AS poi_id
# WHERE 1 = 0
# """)

# try:
#     blacklist = (
#         spark.read
#         .option("header","false")
#         .csv("Files/pipe_poi_google/blacklist.csv")
#         .selectExpr("trim(_c0) as policy_value")
#         .filter("policy_value is not null and policy_value<>''")
#     )
#     blacklist.createOrReplaceTempView("v2_blacklist")

#     # Supports existing poi_id list; if the file is migrated to source_id /
#     # poi_group_id, extend this join.
#     spark.sql("""
#     CREATE OR REPLACE TEMP VIEW v2_blacklist_affected_poi_ids AS
#     SELECT DISTINCT sm2.poi_id
#     FROM poi_source_map sm1
#     JOIN v2_blacklist b
#       ON b.policy_value = sm1.poi_id
#     JOIN poi_source_map sm2
#       ON COALESCE(sm2.poi_group_id,'') = COALESCE(sm1.poi_group_id,'')
#      AND sm2.source_name = sm1.source_name
#      AND sm2.source_id = sm1.source_id
#     WHERE sm2.poi_id IS NOT NULL
#     """)

#     spark.sql("""
#     MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.poi_master tgt
#     USING v2_blacklist_affected_poi_ids src
#       ON tgt.poi_id = src.poi_id
#     WHEN MATCHED THEN UPDATE SET
#       tgt.is_active = false,
#       tgt.updated_at = current_timestamp()
#     """)
# except Exception as e:
#     print(f"Blacklist skipped: {e}")

# # Hard invariants: fail the run before watermark commit if violated.
# critical_count = spark.sql(f"SELECT COUNT(*) c FROM poi_quality_issue WHERE run_id='{run_id}' AND severity='CRITICAL' AND status='OPEN'").first()["c"]
# if critical_count > 0:
#     raise Exception(f"DATA_QUALITY_CRITICAL: {critical_count} critical issues. Watermark must not advance.")

# # NB60 completed. Outbound Shared STG staging removed by design.
# notebookutils.notebook.exit("SUCCESS")
